# EDA — NYT10m

Inspect **training and validation data**, without reading the test set. Objectives: read real examples, build bags by ordered entity pair, inspect multi-label annotations and `NA`, and measure bag sizes.

These observations describe the available annotations: multiple distantly supervised labels do not establish that every sentence expresses all listed relations. This notebook does not select models or thresholds and does not modify the original data.

In [1]:
# Set up imports, verify that this notebook uses the project's virtual environment,
# and define the two allowed data splits. Helper functions format readable tables.
import ast
import hashlib
import html
import json
import platform
import sys
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from IPython.display import display, Markdown, HTML

# Works when Jupyter is launched from the project root or source/.
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / 'data' / 'nyt10m').is_dir()), None
)
assert PROJECT_ROOT is not None, 'Launch the notebook from the Ensembling project.'
assert Path(sys.prefix).resolve() == (PROJECT_ROOT / '.venv').resolve(), (
    'Select the kernel for the project .venv.'
)
DATA_DIR = PROJECT_ROOT / 'data' / 'nyt10m'
SPLITS = ('train', 'val')  # Explicit allowlist: no access to the test set.
INPUTS = {split: DATA_DIR / f'nyt10m_{split}.txt' for split in SPLITS}
LABELS_PATH = DATA_DIR / 'nyt10m_rel2id.json'
rel2id = json.loads(LABELS_PATH.read_text(encoding='utf-8'))
assert 'NA' in rel2id
NA = 'NA'  # Dataset label. Distinct from None, missing data, or the string N/A.

print('Python:', sys.version.split()[0])
print('Kernel:', sys.executable)
print('Input:', ', '.join(INPUTS))
print('Labels:', len(rel2id), '| NA ID:', rel2id[NA])

def table(rows, columns):
    rows = list(rows)
    esc = lambda x: html.escape(str(x))
    header = ''.join(f'<th>{esc(c)}</th>' for c in columns)
    body = ''.join('<tr>' + ''.join(f'<td>{esc(row.get(c, ""))}</td>' for c in columns) + '</tr>' for row in rows)
    display(HTML('<div style="overflow-x:auto"><table><thead><tr>' + header + '</tr></thead><tbody>' + body + '</tbody></table></div>'))

def pct(n, total):
    return f'{100*n/total:.2f}%' if total else 'undefined'

Python: 3.14.3
Kernel: /Users/alessandro/Progetti/Ensembling/.venv/bin/python
Input: train, val
Labels: 25 | NA ID: 0


## 1. Loading, schema, and checks

We use the training and validation splits of NYT10m, a relation-extraction dataset built from New York Times text. The task is to identify the relationship between two specified entities in a sentence—for example, whether a location contains another location. These files contain sentence-level records, rather than complete news articles.
OpenNRE records may also contain **`anno_relation_list`**. This field lists the relations manually verified for an entity pair in the human-annotated evaluation data. It can provide a better reference for evaluation than a single distant-supervision label, which may be noisy. It is optional: it is absent from all records in the training and validation files analyzed here. An empty list, if encountered, means that the field is present with no listed relations; it is different from a missing field and from the dataset label `NA`.

Each line of a split file is a serialized dictionary containing:
- text: the sentence to read.
- h (head) and t (tail): the two entities whose relationship we want to identify. Each includes a name, a knowledge-base id, and a character span pos = [start, end] in the text (end excluded). “Head” and “tail” identify the ordered arguments of the relation, not necessarily the grammatical subject and object.
- relation: the assigned relation label, such as /location/location/contains. NA denotes no positive relation among the dataset's target labels; it is not a missing value.
For example, the first training record mentions New York as the head and Carnegie Hall as the tail, with the label /location/location/contains. The separate file nyt10m_rel2id.json maps relation names to numeric IDs. Training and validation labels come from distant supervision: knowledge-base relations are associated with entity mentions, so an assigned label is not guaranteed to be expressed by that particular sentence.
The same entity pair can appear in several records. We group those records into a bag, allowing us to inspect all available sentences and labels for that pair together.

Each row represents an entity-pair mention in a sentence. The bag key is **`(h.id, t.id)`**: order is preserved because relations may be directional. Grouping is performed separately within each split, without including the relation in the key.

We retain both `relation` and `anno_relation_list`. When the latter is present, we use it as the label set for the analysis; otherwise, we use `relation`. An empty manual annotation list remains empty rather than being converted to `NA`. Counts based on `relation` are also reported separately.

We check the schema, unknown labels, missing fields, and whether entity spans match their names. Anomalies are reported without automatically correcting the data.

In [2]:
# Parse each line, check required fields and entity spans, and report data-quality issues.
# Group records into split-local bags keyed by the ordered head/tail entity IDs.
# Keep relation and optional manual annotations separate; hash inputs for reproducibility.
# Each compact item is (text, head name, tail name, relation, annotations).
# annotations=None means the field is absent; () means a present but empty list.
bags = {} # stores records grouped by (head_id, tail_id) for each split
quality = {} # stores counts of data-quality issues for each split
first_records = {} # stores the first record seen for each split, for inspection
provenance = [] # stores file metadata and SHA256 hash for each split

def parse_record(line):
    try:
        return json.loads(line) # Parse as JSON first, which is the expected format.
    except json.JSONDecodeError:
        return ast.literal_eval(line)

for split, path in INPUTS.items(): # Iterate over the allowed splits and their corresponding input files.
    grouped = defaultdict(list) 
    checks = Counter({name: 0 for name in [
        'blank lines', 'records with anno_relation_list', 'explicit empty annotations',
        'out-of-bounds spans', 'spans differing from names',
        'unknown labels (records)', 'records containing the string N/A',
        'records containing an empty string', 'head and tail with the same ID', 'record'
    ]})
    digest = hashlib.sha256() # Initialize a SHA256 hash object to compute the hash of the file contents for reproducibility.
    with path.open('rb') as file:
        for line_number, raw in enumerate(file, 1): # Read the file line by line, keeping track of the line number for error reporting.
            digest.update(raw)
            if not raw.strip():
                checks['blank lines'] += 1
                continue
            try:
                record = parse_record(raw.decode('utf-8'))
                assert isinstance(record, dict)
                text, h, t = record['text'], record['h'], record['t']
                relation = record['relation']
                assert isinstance(text, str) and isinstance(relation, str)
                assert isinstance(h['id'], str) and h['id']
                assert isinstance(t['id'], str) and t['id']
                annotations = record.get('anno_relation_list')
                if 'anno_relation_list' in record:
                    assert isinstance(annotations, list)
                    assert all(isinstance(label, str) for label in annotations)
                    checks['records with anno_relation_list'] += 1
                    if not annotations:
                        checks['explicit empty annotations'] += 1
                    annotations = tuple(annotations)
                for entity in (h, t):
                    assert isinstance(entity['name'], str)
                    start, end = entity['pos']
                    assert isinstance(start, int) and isinstance(end, int)
                    if not 0 <= start < end <= len(text):
                        checks['out-of-bounds spans'] += 1
                    elif text[start:end] != entity['name']:
                        checks['spans differing from names'] += 1
                labels = {relation} | set(annotations or ())
                checks['unknown labels (records)'] += bool(labels - set(rel2id))
                checks['records containing the string N/A'] += 'N/A' in labels
                checks['records containing an empty string'] += '' in labels
                checks['head and tail with the same ID'] += h['id'] == t['id']
            except Exception as exc:
                raise ValueError(f'{path.name}, line {line_number}: invalid record') from exc
            if split not in first_records:
                first_records[split] = record
            grouped[(h['id'], t['id'])].append(
                (text, h['name'], t['name'], relation, annotations)
            )
            checks['record'] += 1
    bags[split] = dict(grouped)
    quality[split] = checks
    provenance.append({'file': path.name, 'bytes': path.stat().st_size, 'sha256': digest.hexdigest()})
    print(f'{split}: {checks["record"]:,} record → {len(grouped):,} ordered pairs')

check_names = sorted(set().union(*(q.keys() for q in quality.values())))
table([{'check': name, **{s: quality[s][name] for s in SPLITS}} for name in check_names],
      ['check', *SPLITS])

train: 417,893 record → 255,526 ordered pairs
val: 46,422 record → 36,266 ordered pairs


check,train,val
blank lines,0,0
explicit empty annotations,0,0
head and tail with the same ID,0,0
out-of-bounds spans,0,0
record,417893,46422
records containing an empty string,0,0
records containing the string N/A,0,0
records with anno_relation_list,0,0
spans differing from names,0,0
unknown labels (records),0,0


In [3]:
# Print one complete source record from each analyzed split so we can inspect the
# original field names and values before looking at derived statistics.
for split in SPLITS:
    display(Markdown(f'**Complete real example — {split}**'))
    print(json.dumps(first_records[split], ensure_ascii=False, indent=2))

**Complete real example — train**

{
  "text": "One of the newest auditoriums in New York , built into the bedrock below Carnegie Hall , has added a dose of richness to the concert scene .",
  "relation": "/location/location/contains",
  "t": {
    "pos": [
      73,
      86
    ],
    "id": "m.016p8t",
    "name": "Carnegie Hall"
  },
  "h": {
    "pos": [
      33,
      41
    ],
    "id": "m.059rby",
    "name": "New York"
  }
}


**Complete real example — val**

{
  "text": "Michel Ducros , its president and chief , announced earlier this week that he was selling 9 of Fauchon 's 12 satellite shops in Paris to the rival gourmet food chain Lenôtre .",
  "relation": "/location/location/contains",
  "t": {
    "pos": [
      95,
      102
    ],
    "id": "m.0379sy",
    "name": "Fauchon"
  },
  "h": {
    "pos": [
      128,
      133
    ],
    "id": "m.05qtj",
    "name": "Paris"
  }
}


The printed examples show the original serialized records. In this release, `relation` is present in train and validation, while `anno_relation_list` is not; the check table below confirms the counts. Later cells therefore use `relation` for these splits. The optional-field handling documents how to interpret annotated records if you inspect that schema separately.

## 2. Bags, multi-label annotations, and NA

For each pair, we compute the union of the labels across its rows. The following categories are **mutually exclusive**:

- **NA only:** the union is exactly `{NA}`.
- **One positive without NA:** one positive relation and no NA label.
- **Multi-label without NA:** at least two positive relations and no NA label.
- **NA + positives:** NA and one or more positive relations occur in rows of the same bag.
- **No labels:** an empty union.
- **Unknown labels:** at least one label is missing from the relation map.

We separately define **positive multi-label** bags as those with at least two distinct positive relations. This can include bags in the NA + positives category. The coexistence of NA and positive labels warrants inspection, but is not automatically evidence of an error.

In [4]:
# Resolve the effective labels for each row: use manual annotations if present,
# otherwise use the row's relation label. Then classify each pair's combined labels.
# The summary distinguishes NA-only bags from bags that mix NA and positive labels.
def row_labels(row):
    return set(row[4]) if row[4] is not None else {row[3]}

def bag_info(rows):
    labels = set().union(*(row_labels(row) for row in rows))
    positives = labels & (set(rel2id) - {NA})
    if labels - set(rel2id):
        category = 'Unknown labels'
    elif not labels:
        category = 'No labels'
    elif labels == {NA}:
        category = 'NA only'
    elif NA in labels:
        category = 'NA + positives'
    elif len(positives) == 1:
        category = 'One positive without NA'
    else:
        category = 'Multi-label without NA'
    return {'n': len(rows), 'unique_texts': len({r[0] for r in rows}),
            'labels': labels, 'positives': positives, 'category': category}

bag_stats = {s: {key: bag_info(rows) for key, rows in bags[s].items()} for s in SPLITS}
summary = []
for split in SPLITS:
    stats = list(bag_stats[split].values())
    n_bags = len(stats)
    record_na = sum(r[3] == NA for rows in bags[split].values() for r in rows)
    multi = sum(len(b['positives']) >= 2 for b in stats)
    summary.append({'split': split, 'record': quality[split]['record'], 'bag': n_bags,
                    'NA in relation (record)': record_na,
                    '% NA records': pct(record_na, quality[split]['record']),
                    'positive multi-label bags': multi, '% multi-label bags': pct(multi, n_bags)})
table(summary, list(summary[0]))

categories = ['NA only', 'One positive without NA', 'Multi-label without NA',
              'NA + positives', 'No labels', 'Unknown labels']
category_rows = []
for split in SPLITS:
    counts = Counter(b['category'] for b in bag_stats[split].values())
    assert sum(counts.values()) == len(bags[split])
    for category in categories:
        category_rows.append({'split': split, 'category': category, 'bag': counts[category],
                              '% bag': pct(counts[category], len(bags[split]))})
table(category_rows, ['split', 'category', 'bag', '% bag'])

split,record,bag,NA in relation (record),% NA records,positive multi-label bags,% multi-label bags
train,417893,255526,333855,79.89%,1108,0.43%
val,46422,36266,37094,79.91%,164,0.45%


split,category,bag,% bag
train,NA only,239571,93.76%
train,One positive without NA,14846,5.81%
train,Multi-label without NA,1108,0.43%
train,NA + positives,1,0.00%
train,No labels,0,0.00%
train,Unknown labels,0,0.00%
val,NA only,32392,89.32%
val,One positive without NA,3710,10.23%
val,Multi-label without NA,164,0.45%
val,NA + positives,0,0.00%


In [5]:
# Compare raw row-level relation counts with the number of bags carrying each label.
# A bag can carry multiple labels, so bag percentages across labels may exceed 100%.
# A relation can occur in multiple rows of the same bag.
# Bag-level percentages across relations can sum to more than 100%.
label_rows = []
for split in SPLITS:
    raw_counts = Counter(r[3] for rows in bags[split].values() for r in rows)
    bag_counts = Counter(label for b in bag_stats[split].values() for label in b['labels'])
    for label in sorted(set(rel2id) | set(raw_counts) | set(bag_counts), key=lambda x: (-bag_counts[x], x)):
        label_rows.append({'split': split, 'label': label,
                           'record (relation)': raw_counts[label],
                           'bags (effective labels)': bag_counts[label],
                           '% bag': pct(bag_counts[label], len(bags[split]))})
table(label_rows, list(label_rows[0]))

split,label,record (relation),bags (effective labels),% bag
train,NA,333855,239572,93.76%
train,/location/location/contains,40158,7341,2.87%
train,/people/person/nationality,6752,2255,0.88%
train,/people/person/place_lived,6071,2157,0.84%
train,/business/person/company,4761,1335,0.52%
train,/people/person/place_of_birth,2807,1299,0.51%
train,/people/deceasedperson/place_of_death,1742,671,0.26%
train,/location/neighborhood/neighborhood_of,5012,456,0.18%
train,/location/country/administrative_divisions,4441,324,0.13%
train,/location/administrative_division/country,4441,318,0.12%


The first summary counts `NA` on individual records using `relation`, then counts bags by their combined effective labels. These are different units: a bag may contain many records. Multi-label means multiple **distinct positive relation types** across the pair's records.

## 3. Bag sizes and repeated texts

Our primary size measure is the **number of records per bag**. We also count distinct texts: the same sentence may occur more than once with different annotations. We do not remove rows because deduplicating by text alone could discard labels. Percentiles use linear interpolation.

In [6]:
# Compute bag-size percentiles and a size histogram. Also surface the largest bags
# and count repeated sentence texts without removing any source records.
def quantile(values, q):
    values = sorted(values)
    if not values:
        return None
    position = (len(values) - 1) * q
    lo = int(position)
    hi = min(lo + 1, len(values) - 1)
    return round(values[lo] + (values[hi] - values[lo]) * (position - lo), 2)

length_rows = []
for split in SPLITS:
    stats = list(bag_stats[split].values())
    sizes = [b['n'] for b in stats]
    length_rows.append({'split': split, 'min': min(sizes), 'mean': round(sum(sizes)/len(sizes), 2),
                        'p50': quantile(sizes, .5), 'p90': quantile(sizes, .9),
                        'p95': quantile(sizes, .95), 'p99': quantile(sizes, .99), 'max': max(sizes),
                        '% singleton': pct(sum(n == 1 for n in sizes), len(sizes)),
                        'bags with repeated texts': sum(b['unique_texts'] < b['n'] for b in stats),
                        'rows beyond unique texts': sum(b['n'] - b['unique_texts'] for b in stats)})
table(length_rows, list(length_rows[0]))

bins = [(1, 1), (2, 2), (3, 5), (6, 10), (11, 20), (21, 50), (51, 100), (101, float('inf'))]
binned = []
for split in SPLITS:
    sizes = [b['n'] for b in bag_stats[split].values()]
    for low, high in bins:
        n = sum(low <= size <= high for size in sizes)
        binned.append({'split': split, 'records per bag': f'{low}–{high}' if high != float('inf') else '101+',
                       'bag': n, '% bag': pct(n, len(sizes))})
table(binned, ['split', 'records per bag', 'bag', '% bag'])

largest = []
for split in SPLITS:
    for key, info in sorted(bag_stats[split].items(), key=lambda item: (-item[1]['n'], item[0]))[:10]:
        row = bags[split][key][0]
        largest.append({'split': split, 'head': row[1], 'tail': row[2], 'pair IDs': key,
                        'record': info['n'], 'distinct texts': info['unique_texts'],
                        'labels': ', '.join(sorted(info['labels']))})
table(largest, list(largest[0]))

split,min,mean,p50,p90,p95,p99,max,% singleton,bags with repeated texts,rows beyond unique texts
train,1,1.64,1.0,2.0,3.0,10.0,1996,85.85%,1107,11080
val,1,1.28,1.0,1.0,2.0,6.0,224,91.59%,74,156


split,records per bag,bag,% bag
train,1–1,219360,85.85%
train,2–2,19872,7.78%
train,3–5,10669,4.18%
train,6–10,3123,1.22%
train,11–20,1398,0.55%
train,21–50,709,0.28%
train,51–100,244,0.10%
train,101+,151,0.06%
val,1–1,33217,91.59%
val,2–2,1721,4.75%


split,head,tail,pair IDs,record,distinct texts,labels
train,Iraq,Baghdad,"('m.0d05q4', 'm.01fqm')",1996,1095,"/location/country/capital, /location/location/contains"
train,Saddam Hussein,Iraq,"('m.079dy', 'm.0d05q4')",1735,642,"/people/deceasedperson/place_of_death, /people/person/nationality, /people/person/place_of_birth"
train,China,Beijing,"('m.0d05w3', 'm.01914')",1727,642,"/location/country/administrative_divisions, /location/country/capital, /location/location/contains"
train,Russia,Moscow,"('m.06bnz', 'm.04swd')",880,323,"/location/country/administrative_divisions, /location/country/capital, /location/location/contains"
train,Iran,Tehran,"('m.03shp', 'm.0ftlx')",797,444,"/location/country/capital, /location/location/contains"
train,New York,Manhattan,"('m.059rby', 'm.0cc56')",720,720,/location/location/contains
train,Sudan,Darfur,"('m.06tw8', 'm.02vtml')",673,367,"/location/country/administrative_divisions, /location/location/contains"
train,Beijing,China,"('m.01914', 'm.0d05w3')",570,570,/location/administrative_division/country
train,Japan,Tokyo,"('m.03_3d', 'm.07dfk')",553,201,"/location/country/administrative_divisions, /location/country/capital, /location/location/contains"
train,Ohio,Columbus,"('m.05kkh', 'm.01smm')",515,282,"/location/location/contains, /location/region/capital"


These statistics describe how many sentence records are available per entity pair. They help estimate the potential input volume for bag-level experiments; they do not prescribe how many sentences a method should use.

## 4. Real examples of different bag types

Examples are selected deterministically by ID for illustration: **this is not a representative sample**. We prioritize one example per distinct label set, displaying up to eight rows per bag. To inspect other pairs, use `show_bag(split, (head_id, tail_id))`.

In [7]:
# Provide a helper for inspecting all rows for one entity pair, prioritizing distinct
# label sets. Show deterministic examples of each available bag category per split.
def show_bag(split, key, max_examples=8):
    assert split in SPLITS
    rows = bags[split][key]
    info = bag_stats[split][key]
    display(Markdown(f'**{split} — {info["category"]}**'))
    print('Ordered pair:', key)
    print('Head:', rows[0][1], '| Tail:', rows[0][2])
    print('Record:', info['n'], '| Distinct texts:', info['unique_texts'])
    print('Label union:', sorted(info['labels']))
    selected, seen = [], set()
    for i, row in enumerate(rows):
        signature = tuple(sorted(row_labels(row)))
        if signature not in seen:
            selected.append(i)
            seen.add(signature)
    selected = (selected + [i for i in range(len(rows)) if i not in selected])[:max_examples]
    print(f'Showing {len(selected)} / {len(rows)} record; {len(seen)} distinct label sets.')
    for i in selected:
        row = rows[i]
        print(f'\nRecord {i+1} | relation={row[3]} | anno_relation_list={row[4]}')
        print(row[0])

predicates = {
    'NA only': lambda b: b['category'] == 'NA only',
    'One positive without NA': lambda b: b['category'] == 'One positive without NA',
    'At least two positives': lambda b: len(b['positives']) >= 2,
    'NA + positives': lambda b: b['category'] == 'NA + positives',
}
for split in SPLITS:
    for title, predicate in predicates.items():
        candidates = [key for key, b in bag_stats[split].items() if predicate(b)]
        display(Markdown(f'**Selection: {title} ({split})**'))
        if candidates:
            show_bag(split, min(candidates))
        else:
            print('No matching bag in this split.')

**Selection: NA only (train)**

**train — NA only**

Ordered pair: ('m.010016', 'm.0492jkz')
Head: Denton | Tail: Daisy Hill
Record: 1 | Distinct texts: 1
Label union: ['NA']
Showing 1 / 1 record; 1 distinct label sets.

Record 1 | relation=NA | anno_relation_list=None
Daisy Hill , the daughter of May Brawley Hill of Warren , Conn. , and Frederick Hill of Manhattan , was married last evening to Peter Nicholas Sanders , the son of Malena Kuss of Denton , Tex. , and the late Denis Sanders .


**Selection: One positive without NA (train)**

**train — One positive without NA**

...

**Selection: At least two positives (val)**

**val — Multi-label without NA**

Ordered pair: ('m.012x9d', 'm.0947l')
Head: Paolo Maldini | Tail: Milan
Record: 2 | Distinct texts: 2
Label union: ['/people/person/place_lived', '/people/person/place_of_birth']
Showing 2 / 2 record; 2 distinct label sets.

Record 1 | relation=/people/person/place_of_birth | anno_relation_list=None
The class of the organization comes from Milan 's captain , Paolo Maldini , the Derek Jeter of soccer -- great player , great leader , handsome , suave .

Record 2 | relation=/people/person/place_lived | anno_relation_list=None
Paolo Maldini , Milan 's captain , told Reuters : '' It is much better to be able to prepare for a game like this one with seven days of training than to play a tough tie such as ours , but I prefer to play in the Champions League final and I would n't swap that with Juve .


**Selection: NA + positives (val)**

No matching bag in this split.


## 5. Training–validation overlap

We measure the occurrence of the same **ordered entity pairs** and sentences in both splits. Overlap indicates that validation performance may not measure generalization to unseen pairs or sentences. It is not, by itself, evidence of contamination: its relevance depends on the evaluation protocol. We preserve the original splits, do not automatically rebuild them, and do not access the test set.

In [8]:
# Check whether entity pairs or exact sentence texts cross from training into validation.
# Confirm that grouping retained every record, then free temporary text sets from memory.
train_pairs, val_pairs = set(bags['train']), set(bags['val'])
shared_pairs = train_pairs & val_pairs
train_texts = {r[0] for rows in bags['train'].values() for r in rows}
val_texts = {r[0] for rows in bags['val'].values() for r in rows}
shared_texts = train_texts & val_texts
print('Shared pairs:', len(shared_pairs))
print('% validation pairs already in training:', pct(len(shared_pairs), len(val_pairs)))
print('Shared distinct sentences (exact match):', len(shared_texts))
print('% distinct validation sentences already in training:', pct(len(shared_texts), len(val_texts)))
# Grouping consistency: no records lost or added.
for split in SPLITS:
    assert sum(info['n'] for info in bag_stats[split].values()) == quality[split]['record']
print('OK: record and grouping counts are consistent.')
del train_texts, val_texts, shared_texts

Shared pairs: 12043
% validation pairs already in training: 33.21%
Shared distinct sentences (exact match): 25260
% distinct validation sentences already in training: 60.72%
OK: record and grouping counts are consistent.


## 6. Provenance and notes

The hashes identify the exact files analyzed; they are not compared against checksums published by the distributor. Save the notebook after execution to retain its results.

**Questions to document after inspection:**
- How does the record-level NA frequency differ from the percentage of NA-only bags?
- How many bags have at least two positive relations? Do the examples support all their labels?
- How might the long tail of bag sizes and repeated texts affect inference cost?
- What kinds of generalization can be measured given the training–validation overlap?

Choices concerning aggregation, sampling, NA handling, and thresholds should be developed on training/validation data and fixed before final evaluation. Do not infer test-set properties from this EDA.

Sources: [OpenNRE](https://github.com/thunlp/OpenNRE), [download script](https://github.com/thunlp/OpenNRE/blob/master/benchmark/download_nyt10m.sh), [data loader and grouping](https://github.com/thunlp/OpenNRE/blob/master/opennre/framework/data_loader.py).

In [9]:
# Record timestamps, runtime, split names, grouping/label conventions, and file hashes
# so the analysis can be tied to the exact local dataset files.
manifest = {
    'analysis_utc': datetime.now(timezone.utc).isoformat(),
    'python': platform.python_version(),
    'analyzed_splits': list(SPLITS),
    'bag_key': ['h.id', 't.id'],
    'label_source': 'anno_relation_list when present, otherwise relation',
    'file': provenance + [{
        'file': LABELS_PATH.name,
        'bytes': LABELS_PATH.stat().st_size,
        'sha256': hashlib.sha256(LABELS_PATH.read_bytes()).hexdigest(),
    }],
}
print(json.dumps(manifest, ensure_ascii=False, indent=2))

{
  "analysis_utc": "2026-09-24T12:58:31.640253+00:00",
  "python": "3.14.3",
  "analyzed_splits": [
    "train",
    "val"
  ],
  "bag_key": [
    "h.id",
    "t.id"
  ],
  "label_source": "anno_relation_list when present, otherwise relation",
  "file": [
    {
      "file": "nyt10m_train.txt",
      "bytes": 162337081,
      "sha256": "33900650c692279adbe0a041b5c6a90e9216c905c73975905a04669695f7f2af"
    },
    {
      "file": "nyt10m_val.txt",
      "bytes": 18031002,
      "sha256": "614aa5be9dd678d146cf8d44029e4c993f82add560ef7c00dfcb366416bed18c"
    },
    {
      "file": "nyt10m_rel2id.json",
      "bytes": 902,
      "sha256": "c6996ba3f1491bc5ca887fdec2e38f46b81420eab41f805727ebaf5fc757d5c8"
    }
  ]
}
